# Existing analyst approach

Prepared by the fictional operations analytics team. Reproduce the submitted analysis, inspect the code and evaluate the claims in `ANALYST_MEMO.md`. Use the `.venv-py311` kernel.

The submitted implementation is a review subject. Its execution and reported metrics alone do not establish that its conclusions are valid.


Step 1: Defining teh analytical objective

1.1 What business decision are we supporting?
- should the opereations director adopt teh recommendation? Validity and reliability of the method proposed
- reviewing an anlayst's recommednation regarding courier change and predictive late-delivery alert

1.2 What does one row represent? 
- one dispatched shipment

1.3 Target; what outcome are we predicting or explaining?
- explaining possible causes of the late-delivery
- predicting shipment that will miss their promised transit time

1.4 Timing: What information would be available when making decision? 
- 
| shipment_id | Shipment identifier |
| seller_id | Merchant identifier, known at dispatch |
| dispatch_date | Dispatch date, January through April 2026 |
| carrier | Standard or Express service selected at dispatch |
| route_type | Urban or Rural, known at dispatch |
| promised_transit_days | Contractual maximum transit time in days, known at dispatch |
| distance_km | Estimated route distance available at dispatch, sometimes missing |
| warehouse_backlog | Warehouse workload at dispatch, sometimes missing |
<!-- | actual_transit_days | Final realised transit time, recorded upon delivery | -->
<!-- | exception_scans_after_dispatch | Count accumulated over the delivery journey, available in final form after delivery | -->
<!-- | delivered_date | Final delivery date when observed | -->
<!-- | late_delivery | 1 if final transit exceeds the promise, 0 otherwise, blank if final outcome remains unresolved | -->

In [2]:
import pandas as pd
import numpy as np

shipments = pd.read_csv("data/shipments.csv")
rates = pd.read_csv("data/rate_history.csv")
print("Shipment export:", shipments.shape)
print("Rate records:", rates.shape)


Shipment export: (1800, 12)
Rate records: (8, 4)


In [3]:
shipments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   shipment_id                     1800 non-null   object 
 1   seller_id                       1800 non-null   object 
 2   dispatch_date                   1800 non-null   object 
 3   carrier                         1800 non-null   object 
 4   route_type                      1800 non-null   object 
 5   promised_transit_days           1800 non-null   int64  
 6   distance_km                     1745 non-null   float64
 7   warehouse_backlog               1757 non-null   float64
 8   actual_transit_days             1744 non-null   float64
 9   exception_scans_after_dispatch  1744 non-null   float64
 10  delivered_date                  1744 non-null   object 
 11  late_delivery                   1744 non-null   float64
dtypes: float64(5), int64(1), object(6)

In [4]:
print(shipments.describe(include=["object", "category"]))

       shipment_id seller_id dispatch_date   carrier route_type delivered_date
count         1800      1800          1800      1800       1800           1744
unique        1800       120           120         2          2            119
top         S00001      V047    2026-04-17  Standard      Urban     2026-03-22
freq             1        24            23      1100       1047             30


In [5]:
print(shipments.describe())


       promised_transit_days  distance_km  warehouse_backlog  \
count            1800.000000  1745.000000        1757.000000   
mean                3.255000   128.363954          19.426295   
std                 1.480268    99.002023           5.509967   
min                 2.000000     5.000000           5.000000   
25%                 2.000000    40.100000          15.000000   
50%                 2.000000    82.700000          19.000000   
75%                 5.000000   232.600000          23.000000   
max                 5.000000   325.000000          39.000000   

       actual_transit_days  exception_scans_after_dispatch  late_delivery  
count          1744.000000                     1744.000000    1744.000000  
mean              3.335436                        0.719610       0.239106  
std               2.580963                        1.264705       0.426660  
min               1.000000                        0.000000       0.000000  
25%               1.000000                 

In [6]:
shipments.isna().sum()

shipment_id                        0
seller_id                          0
dispatch_date                      0
carrier                            0
route_type                         0
promised_transit_days              0
distance_km                       55
warehouse_backlog                 43
actual_transit_days               56
exception_scans_after_dispatch    56
delivered_date                    56
late_delivery                     56
dtype: int64

#record rows with na before conversion


In [ ]:
# good practices to save raw columns
shipments_raw = shipments.copy()

missing_delivered_date = shipments[shipments["delivered_date"].isna()]

# convert delivered_date and dispatch_date to date type

date_cols = ["delivered_date", "dispatch_date"]

# for col in date_cols:
#     shipments[col] = pd.date_time(
#         shipments[col],
#         errors="coerce"
#     )

shipments[date_cols] = shipments[date_cols].apply(
    pd.to_datetime,
    format="%Y-%m-%d",
    errors="coerce"
)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   shipment_id                     1800 non-null   object        
 1   seller_id                       1800 non-null   object        
 2   dispatch_date                   1800 non-null   datetime64[ns]
 3   carrier                         1800 non-null   object        
 4   route_type                      1800 non-null   object        
 5   promised_transit_days           1800 non-null   int64         
 6   distance_km                     1745 non-null   float64       
 7   warehouse_backlog               1757 non-null   float64       
 8   actual_transit_days             1744 non-null   float64       
 9   exception_scans_after_dispatch  1744 non-null   float64       
 10  delivered_date                  1744 non-null   datetime64[ns]
 11  late

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


df = shipments.copy()

profile = pd.DataFrame({
    "dtype": df.dtypes,
    "unique": df.nunique(dropna=False),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2)
})

display(profile)


,dtype,unique,missing,missing_pct
shipment_id,object,1800,0,0.00
seller_id,object,120,0,0.00
dispatch_date,object,120,0,0.00
carrier,object,2,0,0.00
route_type,object,2,0,0.00
promised_transit_days,int64,2,0,0.00
distance_km,float64,1131,55,3.06
warehouse_backlog,float64,35,43,2.39
actual_transit_days,float64,11,56,3.11
exception_scans_after_dispatch,float64,10,56,3.11


In [ ]:
print(shipments.isna().sum())

shipment_id                        0
seller_id                          0
dispatch_date                      0
carrier                            0
route_type                         0
promised_transit_days              0
distance_km                       55
warehouse_backlog                 43
actual_transit_days               56
exception_scans_after_dispatch    56
delivered_date                    56
late_delivery                     56
dtype: int64


Business rules

- [promised_transit_days, warehouse_backlog, actual_transit_days, exception_scans_after_dispatch] >=0 and is whole number
- [distance_kn] > 0
- [shipment_id] should be unique
- assert carrier.str.lower().str.strip().str.title().isin("Express", "Standard")
- assert route_type.str.lower().str.strip().str.title().isin("Urban", "Rural")
- dispatch_date <= delivered_date
- late_delivery = 1 if actual_transit_days > promised_transit_days
- dispatch_date + actual_transit_days = delivered_date
- late_delivery.isin[0, 1]


In [ ]:
assert (shipments[["promised_transit_days", "warehouse_backlog", "actual_transit_days", "exception_scans_after_dispatch"]] >= 0).all().all() 
assert (shipments[["promised_transit_days", "warehouse_backlog", "actual_transit_days", "exception_scans_after_dispatch"]] % 1 == 0).all().all()
assert (shipments["distance_kn"] > 0).all()
assert shipments["shipment_id"].is_unique

assert (shipments["carrier"].str.lower().str.strip().str.title().isin(["Express", "Standard"])).all()
assert (shipments["route_type"].str.lower().str.strip().str.title().isin(["Urban", "Rural"])).all()

assert (shipments["dispatch_date"] <= shipments["delivered_date"]).all()
assert (shipments.loc[(shipments["actual_transit_days"] > shipments["promised_transit_days"]), "late_delivery"] == 1 ).all()

assert (shipments["dispatch_date"] + 
        pd.to_timedelta(shipments["actual_transit_days"], unit="D")                        
        == shipments["delivered_date"]).all()
assert (shipments["late_delivery"].isin([0, 1])).all()


In [ ]:
print(shipments.nunique().sort_values())
print(shipments.nunique(dropna=False).sort_values())

# Issue 1 
carrier is not unique, so merge creates many to many joins, inflating the rows 



In [14]:
# Enrich shipments with available partner pricing.
analysis = shipments.merge(rates, on="carrier", how="left")
analysis["late_delivery"] = analysis["late_delivery"].fillna(0)

carrier_summary = analysis.groupby("carrier").agg(
    shipments=("shipment_id", "count"),
    late_rate=("late_delivery", "mean"),
    total_fees=("fee_cu", "sum")
)
print(carrier_summary.to_string())
print("Estimated total fees:", analysis["fee_cu"].sum())

print(len(analysis), " ", len(shipments))

          shipments  late_rate  total_fees
carrier                                   
Express        2800   0.278571     32340.0
Standard       4400   0.201818     29040.0
Estimated total fees: 61380.0
7200   1800


In [ ]:
from sklearn.model_selection import train_test_split

numeric_features = [
    "distance_km", "warehouse_backlog", "promised_transit_days",
    "actual_transit_days", "exception_scans_after_dispatch", "fee_cu"
]
categorical_features = ["carrier", "route_type_x"]
X = analysis[numeric_features + categorical_features]
y = analysis["late_delivery"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train/test rows:", len(X_train), len(X_test))


Train/test rows: 5760 1440


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

numeric = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric, numeric_features),
    ("cat", categorical, categorical_features)
])
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Model accuracy:", accuracy_score(y_test, predictions))
print("Baseline accuracy:", accuracy_score(y_test, baseline.predict(X_test)))
print("Confusion matrix, labels 0 then 1:")
print(confusion_matrix(y_test, predictions, labels=[0, 1]))
print(classification_report(y_test, predictions, zero_division=0))


Model accuracy: 1.0
Baseline accuracy: 0.7680555555555556
Confusion matrix, labels 0 then 1:
[[1106    0]
 [   0  334]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1106
           1       1.00      1.00      1.00       334

    accuracy                           1.00      1440
   macro avg       1.00      1.00      1.00      1440
weighted avg       1.00      1.00      1.00      1440

